# Alto risco, cobertura territorial e previsão

Este case de **Data Science** investiga onde se concentram registros classificados como alto risco, como se relacionam com a proximidade dos totens e com que precisão prever seu volume diário para 30 dias.

**A execução padrão usa exclusivamente dados sintéticos**, com semente 42, inclusive a geografia e os equipamentos. O estudo privado usa o acervo minimizado e verificado. As funções de processamento e modelagem são importáveis; este notebook organiza a investigação e a comunicação.

Minha contribuição abrange toda a estrutura de engenharia e analytics: preparação segura, qualidade, referências, análise espacial, previsão e comunicação.

Unidade: um protocolo consolidado. Alvo: classificação de risco da fonte, com nível alto ou superior. Data: criação do registro, no fuso `America/Porto_Velho`. O indicador composto de violência/intervenção do pipeline anterior não define este alvo.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
from totens.portfolio.case import run_case, load_case

# Para ler uma execução privada existente, informe seu diretório explicitamente
# por TOTENS_PORTFOLIO_RUN. Não existe seleção automática da base real.
existing_run = os.getenv('TOTENS_PORTFOLIO_RUN')
RUN_DIR = Path(existing_run).resolve() if existing_run else run_case(mode='synthetic', project=PROJECT_ROOT)
result = load_case(RUN_DIR, project=PROJECT_ROOT)
display(Markdown(f"**Modo: {result['mode']} · execução: {result['run_id']}**"))

## Qualidade, repetição e elegibilidade

Repetições são comparadas pelo hash canônico de cada registro minimizado. Não se presume que o último arquivo contenha a versão correta. Protocolos com data ou risco conflitantes ficam fora da análise temporal principal; conflitos de coordenadas afetam a análise espacial. Classificação desconhecida permanece desconhecida.

Os denominadores abaixo conciliam linhas de entrada, protocolos distintos, exclusões e alvo. A tabela de reconciliação histórica compara multiplicidades por protocolo pseudonimizado e instante UTC; uma associação com feriados não prova a causa histórica da diferença.

A reconciliação é uma evidência histórica autenticada, não recalculada pela leitura do legado. Cadastros revisados preservam UUIDs e distinguem procedência, vinculação institucional e localização geométrica de identidade física comprovada.


In [ ]:
quality = result['quality']
display(pd.DataFrame([{k: quality[k] for k in ['input_rows', 'distinct_protocols', 'excess_rows', 'identical_excess_rows', 'different_excess_rows']}]))
display(pd.Series(quality['partition'], name='protocolos').to_frame())
display(Image(filename=str(RUN_DIR / 'figures/quality.png')))
display(Markdown(str(result['legacy_reconciliation'])))
display(Markdown('Referências desta execução:'))
display(pd.Series(result.get('references') or {}, name='referencias').to_frame())


## Exploração temporal

As janelas declaradas dos arquivos não comprovam completude da extração. Um dia com zero no calendário representa zero de registros observados no acervo. Lacunas identificadas são ausentes. A série modelada termina no último mês completo. A comparação de mudanças mensais usa somente meses adjacentes completos. Picos são sinalizados e preservados; sua interpretação depende de investigar cobertura, origem e atualizações.

In [ ]:
daily = pd.read_csv(RUN_DIR / 'daily.csv', parse_dates=['date'])
display(pd.DataFrame([{k: result['temporal'][k] for k in ['start_date', 'end_date', 'calendar_days', 'declared_days', 'gap_days', 'partial_months']}]))
display(pd.Series(result['temporal']['large_monthly_changes'], name='razão entre meses completos').to_frame())
display(Image(filename=str(RUN_DIR / 'figures/monthly.png')))

## Concentração, geografia e cobertura

A seleção espacial exige coordenadas concordantes e um ponto no polígono de Rondônia, incluindo a fronteira. Na demonstração sintética, o polígono é uma área fictícia. O cadastro disponível define proximidade geométrica: datas de operação por equipamento não estão comprovadas.

A grade de referência tem células de 1 km em projeção equivalente centrada na região. DBSCAN usa distância Haversine e pesos de coordenadas coincidentes, preservando o volume sem expandir repetições. A configuração 150 m / 50 registros foi fixada antes dos resultados; nove combinações verificam sensibilidade. A comparação de semestres usa sobreposição das células ocupadas por pontos agrupados, sem exigir igualdade dos rótulos de cluster.

In [ ]:
spatial = result['spatial']
display(pd.DataFrame([{k: spatial[k] for k in ['high_risk_protocols', 'high_risk_coordinate_invalid_or_conflicting', 'high_risk_outside_ro', 'spatial_eligible', 'window_start', 'window_end', 'window_records']}]))
display(pd.DataFrame(spatial['coverage']).T)
display(Markdown('Proximidade somente na janela de 12 meses:'))
display(pd.DataFrame(spatial['window_coverage']).T)
display(Image(filename=str(RUN_DIR / 'figures/coverage.png')))
display(Image(filename=str(RUN_DIR / 'figures/spatial.png')))
display(pd.read_csv(RUN_DIR / 'cluster_sensitivity.csv'))
display(pd.read_csv(RUN_DIR / 'semester_comparison.csv'))
display(Markdown(f"Persistência das células agrupadas entre semestres: **Jaccard = {spatial['semester_clustered_grid_jaccard']}**. Volumes semestrais diferentes e o mínimo fixo de registros também afetam essa medida."))

## Previsão: hipóteses e desenho de avaliação

Prevemos a contagem diária de protocolos elegíveis de alto risco. Os métodos são repetição semanal, média por dia da semana dos últimos 84 dias e boosting Poisson em quatro configurações previamente fixadas.

Os atributos são lags de 1/7/14/28 dias, médias móveis de 7/28 dias calculadas com deslocamento e calendário conhecido. O horizonte de 30 dias é recursivo: previsões anteriores substituem os valores futuros desconhecidos nos lags.

Há seis origens de validação, treino crescente e mínimo de 365 dias utilizáveis. O modelo é selecionado por MAE médio da validação e congelado antes dos três blocos finais de teste (90 dias). A validação interna aleatória do boosting está desativada. MAE e RMSE mantêm a unidade de registros/dia; MASE usa a variação sazonal de sete dias do treinamento e fica indefinido quando essa escala é zero.

O modelo principal não incorpora feriados. O experimento adicional nacional/estadual usa regras da biblioteca e somente validação. O calendário municipal revisado permanece separado, com vigência, alcance administrativo e lacunas explícitas.


In [ ]:
forecast = result['forecast']
display(pd.DataFrame([{k: forecast[k] for k in ['series_start', 'series_end', 'test_start', 'test_end', 'selected_model', 'selected_test_mae', 'best_baseline_test_mae']}]))
display(pd.read_csv(RUN_DIR / 'forecast_performance.csv'))
display(Image(filename=str(RUN_DIR / 'figures/forecast_comparison.png')))
display(Image(filename=str(RUN_DIR / 'figures/forecast_test.png')))
display(Image(filename=str(RUN_DIR / 'figures/forecast_horizon.png')))

## Interpretação, limites e reprodução

A avaliação usa um snapshot consolidado retrospectivo. A disponibilidade das classificações de risco no instante de cada origem histórica não foi comprovada. A projeção após o corte histórico não representa previsão para a data atual; não foram estimados intervalos probabilísticos.

A leitura dos resultados considera classificação, cobertura, atualizações e localização. Clusters são áreas para investigação. Os dados e métodos não estabelecem efeito causal de totens ou estimativas de ocorrências evitadas.

Cada execução possui manifesto com versão, estado do Git, hashes dos insumos, parâmetros e artefatos. Os testes alteram deliberadamente o futuro para verificar que features e previsões de uma origem anterior permanecem iguais. A versão rastreada deste notebook é mantida sem outputs; resultados reais e mapas detalhados ficam privados.

Referências: [DBSCAN e controle de memória](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html), [features defasadas e avaliação temporal](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).

In [ ]:
conclusion = ('O candidato selecionado teve erro maior que o baseline no teste; não há ganho consistente demonstrado.'
              if forecast['selected_test_mae'] > forecast['best_baseline_test_mae']
              else 'O candidato selecionado permaneceu competitivo no teste, dentro deste acervo retrospectivo.')
display(Markdown(f"""
### Síntese dos resultados medidos

- Alto risco da fonte: **{quality['high_risk_protocols']:,} protocolos** após consolidação e exclusões.
- Proximidade de 300 m: **{spatial['coverage']['300']['fraction']:.1%}** do conjunto espacial elegível de {spatial['spatial_eligible']:,} protocolos.
- Modelo selecionado na validação: **{forecast['selected_model']}**. MAE no teste: **{forecast['selected_test_mae']:.3f} registros/dia**, comparado a {forecast['best_baseline_test_mae']:.3f} do melhor baseline no teste.

{conclusion}

O teste final descreve desempenho fora do treinamento e não muda a seleção de modelo. Os denominadores e as exclusões condicionam a interpretação espacial. O relatório completo está em `report.md` no diretório desta execução.
"""))